# 03. SAE-фичи

Этап 2. Прогоняем активации через SAE из Gemma Scope (слои 6, 12, 20): качество реконструкции и то, насколько те же фичи срабатывают на тех же токенах, что и в `fp16`.

Нужны результаты ноутбука 02 с теми же `LIMIT` и `RESULTS`. `fp16` идёт первой.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, sys
REPO = "https://github.com/Kaban17/qsi.git"
if not os.path.exists("/content/qsi"):
    !git clone -q {REPO} /content/qsi
else:
    !git -C /content/qsi pull -q
%cd /content/qsi
!pip install -q -e . bitsandbytes accelerate
sys.path.insert(0, "/content/qsi")

In [ ]:
# Нужен секрет Colab с именем HF_TOKEN и принятая лицензия Gemma на Hugging Face.
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
from pathlib import Path
from transformers import AutoTokenizer
from qsi.data import load_splits, load_wikitext_chunks
from qsi.models import CONFIG_NAMES, MODEL_ID
from qsi.pipeline import truncate_splits
from qsi.results import read_metrics

# FULL = False: дымовой прогон на 5 промптах и двух конфигурациях (несколько минут).
# FULL = True: полный прогон всех 8 конфигураций. Значение должно совпадать во всех трёх ноутбуках.
FULL = False

LIMIT = None if FULL else 5
CONFIGS = CONFIG_NAMES if FULL else ["fp16", "rtn-4"]
RESULTS = Path("/content/drive/MyDrive/qsi") / ("results" if FULL else "results_smoke")

splits = truncate_splits(load_splits(), LIMIT)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
wikitext = load_wikitext_chunks(tokenizer, n_chunks=4 if LIMIT else 40)
{k: len(v) for k, v in splits.items()}, wikitext.shape

In [ ]:
from qsi.pipeline import run_stage2

run_stage2("fp16", RESULTS, splits, wikitext)
sae_fp16 = read_metrics(RESULTS / "fp16")["sae"]
print({layer: round(v["fvu"], 3) for layer, v in sae_fp16.items() if layer.isdigit()})
print("transfer_ok:", sae_fp16["transfer_ok"])

**Проверка переноса.** Эти SAE обучены на базовой `gemma-2-2b`, а мы применяем их к `-it`. Если `transfer_ok` равно `False` (FVU выше 0.5 на всех слоях), остановитесь: метрики фич на такой реконструкции не имеют смысла, и нужен запасной план из спецификации.

In [ ]:
import time
assert sae_fp16["transfer_ok"], "SAE не переносятся на -it: см. пояснение выше"
for name in CONFIGS[1:]:
    start = time.time()
    run_stage2(name, RESULTS, splits, wikitext)
    print(f"{name}: {time.time() - start:.0f} s")

In [ ]:
import pandas as pd
rows = {}
for name in CONFIGS:
    for layer, v in read_metrics(RESULTS / name)["sae"].items():
        if layer.isdigit():
            rows[(name, int(layer))] = {k: v.get(k) for k in ("fvu", "l0", "median_corr", "frac_corr_gt_0.9", "n_dead", "n_new")}
pd.DataFrame(rows).T